
# DSS-LVR IAF regression diagnostics

This notebook has two separate purposes:

1. **Current controlled training:** use an independent RNG for `q0.loc` jitter, isolate diagnostic sampling from the training RNG, and record how the transported posterior changes with flow depth.
2. **Historical IAF reproduction:** reproduce the original `iaf_ordering_depth_test` conditions as closely as possible, including its old RNG behavior, so changes made after that benchmark can be detected.

The historical section is intentionally isolated. Do not use its legacy RNG behavior for new experiments.


In [1]:
from pathlib import Path
import inspect
import math
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from IPython.display import display


def find_project_root():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / "Python" / "model2.py").exists() and (root / "Python" / "bnn_metric.py").exists():
            return root
    raise FileNotFoundError("Run this notebook inside the NFlow repository.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import Python.model2 as md
import Python.bnn_metric as metric

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
HAS_SLAB_INIT = "slab_init" in inspect.signature(md.GroupedBNNVI).parameters

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
print("project root :", ROOT)
print("device       :", DEVICE)
print("slab_init API:", HAS_SLAB_INIT)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
project root : D:\positron\NFlow
device       : cpu
slab_init API: True


## Current simulator

In [6]:
def simfun(
    n=600,
    p=100,
    n_active=10,
    n_true_units=10,
    activation="relu",          # "none", "relu", or "trig"
    structure="mixed",          # "axis" or "mixed"
    features_per_unit=3,        # only used for mixed
    n_interactions=0,           # 0, 1, 2, 3
    n_quadratic=0,              # 0, 1, 2, 3
    extra_scale=0.5,
    sigma2=1.0,
    target_signal_sd=1.5,
    x_low=-2.5,
    x_high=2.5,
    seed=123,
    device=None,
    dtype=torch.float32,
):
    device = torch.device("cpu") if device is None else torch.device(device)
    rng = np.random.default_rng(seed)
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    n, p, n_active, K = int(n), int(p), int(n_active), int(n_true_units)
    n_interactions, n_quadratic = int(n_interactions), int(n_quadratic)

    if activation not in {"none", "relu", "trig"}:
        raise ValueError("activation must be 'none', 'relu', or 'trig'.")
    if structure not in {"axis", "mixed"}:
        raise ValueError("structure must be 'axis' or 'mixed'.")
    if not 0 <= n_interactions <= 3 or not 0 <= n_quadratic <= 3:
        raise ValueError("n_interactions and n_quadratic must be between 0 and 3.")

    X = float(x_low) + (float(x_high) - float(x_low)) * torch.rand(
        n, p, generator=gen, device=device, dtype=dtype
    )

    active_idx = np.sort(rng.choice(p, size=n_active, replace=False))
    active_t = torch.as_tensor(active_idx, device=device, dtype=torch.long)
    Xa = X.index_select(1, active_t)

    W = np.zeros((K, n_active), dtype=np.float32)

    if structure == "axis":
        if K < n_active:
            raise ValueError("axis requires n_true_units >= n_active.")
        supports = [{unit % n_active} for unit in range(K)]
    else:
        m = min(int(features_per_unit), n_active)
        if K * m < n_active:
            raise ValueError("mixed requires n_true_units * features_per_unit >= n_active.")

        supports = [set() for _ in range(K)]
        for pos, feature in enumerate(rng.permutation(n_active)):
            supports[pos % K].add(int(feature))
        for unit in range(K):
            while len(supports[unit]) < m:
                supports[unit].add(int(rng.integers(n_active)))

    for unit, support in enumerate(supports):
        idx = np.asarray(sorted(support), dtype=int)
        signs = rng.choice([-1.0, 1.0], size=len(idx))
        magnitude = rng.uniform(0.8, 1.2)
        W[unit, idx] = magnitude * signs / np.sqrt(len(idx))

    if activation in {"relu", "trig"}:
        bias = rng.uniform(-0.8, 0.8, size=K).astype(np.float32)
    else:
        bias = np.zeros(K, dtype=np.float32)

    amplitude = rng.uniform(0.8, 1.2, size=K).astype(np.float32)

    Wt = torch.as_tensor(W, device=device, dtype=dtype)
    bt = torch.as_tensor(bias, device=device, dtype=dtype)
    at = torch.as_tensor(amplitude, device=device, dtype=dtype)

    pre = Xa @ Wt.T - bt

    trig_functions = []
    if activation == "relu":
        hidden = F.relu(pre)
    elif activation == "trig":
        hidden = torch.empty_like(pre)
        trig_names = ("sin", "cos", "sin2", "cos2")
        for unit in range(K):
            mode = unit % 4
            z = pre[:, unit]
            if mode == 0:
                hidden[:, unit] = torch.sin(z)
            elif mode == 1:
                hidden[:, unit] = torch.cos(z)
            elif mode == 2:
                hidden[:, unit] = torch.sin(z).square()
            else:
                hidden[:, unit] = torch.cos(z).square()
            trig_functions.append(trig_names[mode])
    else:
        hidden = pre

    signal = hidden @ at

    interaction_pairs = []
    if n_interactions > 0:
        candidates = [(j, k) for j in range(n_active) for k in range(j + 1, n_active)]
        rng.shuffle(candidates)
        interaction_pairs = candidates[:n_interactions]

        for j, k in interaction_pairs:
            term = Xa[:, j] * Xa[:, k]
            term = (term - term.mean()) / term.std(unbiased=False).clamp_min(1e-8)
            signal = signal + float(extra_scale) * term

    quadratic_features = []
    if n_quadratic > 0:
        quadratic_features = rng.choice(
            n_active, size=min(n_quadratic, n_active), replace=False
        ).tolist()

        for j in quadratic_features:
            term = Xa[:, j].square()
            term = (term - term.mean()) / term.std(unbiased=False).clamp_min(1e-8)
            signal = signal + float(extra_scale) * term

    signal = signal - signal.mean()
    signal = signal * float(target_signal_sd) / signal.std(unbiased=False).clamp_min(1e-8)

    y = signal + math.sqrt(float(sigma2)) * torch.randn(
        n, generator=gen, device=device, dtype=dtype
    )

    feature_true = torch.zeros(p, device=device, dtype=dtype)
    feature_true[active_t] = 1.0

    info = {
        "sim": f"{structure}_{activation}",
        "seed": int(seed),
        "n": n,
        "p": p,
        "n_active": n_active,
        "active_idx": active_idx,
        "feature_true": feature_true.detach().cpu().numpy(),
        "n_true_units": K,
        "activation": activation,
        "structure": structure,
        "features_per_unit": 1 if structure == "axis" else int(features_per_unit),
        "teacher_supports": [
            active_idx[np.asarray(sorted(s), dtype=int)].tolist()
            for s in supports
        ],
        "trig_functions": trig_functions,
        "interaction_pairs": [
            (int(active_idx[j]), int(active_idx[k]))
            for j, k in interaction_pairs
        ],
        "quadratic_features": [
            int(active_idx[j])
            for j in quadratic_features
        ],
        "n_interactions": len(interaction_pairs),
        "n_quadratic": len(quadratic_features),
        "sigma2": float(sigma2),
        "signal_sd": float(signal.std(unbiased=False)),
    }

    return X, y, feature_true, signal, info


## Controlled trainer with flow-depth diagnostics

The production path below always uses an **independent jitter RNG**. Diagnostic draws also run inside `torch.random.fork_rng`, so changing `record_every` or `R_diag` cannot alter subsequent optimization draws.

Recorded checkpoints distinguish the base posterior (`q0_*_sd`) from the transported posterior (`post_*_sd`), and separately track all-on representation R² (`repr_r2`) versus the actually gated network (`gated_r2`).


In [3]:
def train_trace(
    X_train, y_train, X_eval, signal_eval, X_test, signal_test, *, truth,
    selection_mode="feature_group", hidden_dims=(20,), sigma2=1.0,
    init_sd=0.5, K_flow=4, flow_hidden_units=128, flow_hidden_layers=2,
    scale_clip=2.0, flow_seed=123, iaf_ordering_scheme="cyclic3",
    iaf_shuffle_within_role=True, gate_type="normalized_requ", gate_scale=1.0,
    epochs=1200, warmup_epochs=300, lr=3e-4, R_train=32, R_diag=128,
    R_final=500, record_every=100, init_loc_jitter=0.05, jitter_seed_offset=99173,
    grad_clip=5.0, support_threshold=0.5, seed=123,
    slab_init="auto", slab_sd_ratio=0.1, slab_bias_sd=0.02,
):
    random.seed(int(seed)); np.random.seed(int(seed)); torch.manual_seed(int(seed))
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))

    model_kwargs = dict(
        X=X_train, y=y_train, input_dim=X_train.shape[1], hidden_dims=tuple(hidden_dims), out_dim=1,
        selection_mode=selection_mode, family="gaussian", sigma2=float(sigma2), init_sd=float(init_sd),
        K_flow=int(K_flow), flow_type="iaf", flow_hidden_units=int(flow_hidden_units),
        flow_hidden_layers=int(flow_hidden_layers), scale_clip=float(scale_clip), flow_seed=int(flow_seed),
        iaf_ordering_scheme=iaf_ordering_scheme, iaf_shuffle_within_role=bool(iaf_shuffle_within_role),
        gate_type=gate_type, gate_scale=float(gate_scale),
    )
    if HAS_SLAB_INIT:
        model_kwargs.update(slab_init=slab_init, slab_sd_ratio=slab_sd_ratio, slab_bias_sd=slab_bias_sd)
    model = md.GroupedBNNVI(**model_kwargs).to(DEVICE)

    # Independent jitter RNG: changing K_flow no longer changes q0.loc through RNG consumption.
    if float(init_loc_jitter) > 0:
        g = torch.Generator(device=model.q0.loc.device)
        g.manual_seed(int(seed) + int(jitter_seed_offset))
        jitter = torch.randn(model.q0.loc.shape, generator=g, device=model.q0.loc.device, dtype=model.q0.loc.dtype)
        with torch.no_grad(): model.q0.loc.add_(float(init_loc_jitter) * jitter)

    init_q0_loc = model.q0.loc.detach().cpu().clone()
    init_q0_sd = torch.exp(model.q0.raw_log_scale.detach().clamp(-8.0, 2.0)).cpu().clone()
    optimizer = torch.optim.Adam(model.parameters(), lr=float(lr))
    history = []

    def record(epoch, phase):
        model.eval()
        devices = [torch.cuda.current_device()] if DEVICE.type == "cuda" else []
        with torch.random.fork_rng(devices=devices):
            torch.manual_seed(int(seed) + 700000 + int(epoch))
            if DEVICE.type == "cuda": torch.cuda.manual_seed_all(int(seed) + 700000 + int(epoch))
            with torch.no_grad():
                xi, log_q = model.sample_posterior(int(R_diag))
                s, v, t = model.decoder.split_latent(xi)
                sem = model.decoder.group_semantics(xi)
                pred_repr = model.decoder(X_eval, xi, force_all_on=True)
                pred_gated = model.decoder(X_eval, xi, force_all_on=(phase in {"init", "repr"}))
                f_repr = metric.function_metrics(signal_eval, pred_repr)
                f_gated = metric.function_metrics(signal_eval, pred_gated)
                ll = model.log_likelihood(xi, force_all_on=(phase in {"init", "repr"})).mean()
                prior = model.log_prior(xi).mean()
                q0_sd = torch.exp(model.q0.raw_log_scale.detach().clamp(-5.0, 2.0))
                row = {
                    "epoch": int(epoch), "phase": phase,
                    "repr_r2": float(f_repr["r2"]), "gated_r2": float(f_gated["r2"]),
                    "repr_mse": float(f_repr["mse"]),
                    "pred_var": float(pred_repr.var(dim=0, unbiased=False).mean()),
                    "pred_mean_sd": float(pred_repr.mean(dim=0).std(unbiased=False)),
                    "post_U_sd": float(s.std(dim=0, unbiased=False).mean()),
                    "post_V_sd": float(v.std(dim=0, unbiased=False).mean()),
                    "post_tau_sd": float(t.std(dim=0, unbiased=False).mean()),
                    "q0_U_sd": float(q0_sd[:model.decoder.s_dim].mean()),
                    "q0_V_sd": float(q0_sd[model.decoder.s_dim:model.decoder.s_dim + model.decoder.u_dim].mean()),
                    "q0_tau_sd": float(q0_sd[model.decoder.s_dim + model.decoder.u_dim:].mean()),
                    "margin_mean": float(sem["margin"].mean()), "margin_sd": float(sem["margin"].std(unbiased=False)),
                    "active_rate": float(sem["active"].float().mean()),
                    "loglik": float(ll), "logprior": float(prior), "logq": float(log_q.mean()),
                    "kl_like": float((log_q.mean() - prior)),
                }
                if model.decoder.has_feature_gates:
                    fp = model.decoder.feature_semantics(xi)["active"].float().mean(0)
                    target = torch.as_tensor(np.asarray(truth["feature_true"]) > 0.5, device=fp.device)
                    row["feature_pip_mean"] = float(fp.mean())
                    row["active_feature_pip"] = float(fp[target].mean()) if target.any() else np.nan
                    row["inactive_feature_pip"] = float(fp[~target].mean()) if (~target).any() else np.nan
                if model.decoder.has_unit_gates:
                    up = model.decoder.unit_semantics(xi)["active"].float().mean(0)
                    row["unit_pip_mean"] = float(up.mean())
                    row["expected_units"] = float(up.sum())
        history.append(row)
        print(f"epoch={epoch:04d} {phase:6s} reprR2={row['repr_r2']:+.4f} gatedR2={row['gated_r2']:+.4f} "
              f"predVar={row['pred_var']:.3g} U/V/tauSD={row['post_U_sd']:.3g}/{row['post_V_sd']:.3g}/{row['post_tau_sd']:.3g} "
              f"active={row['active_rate']:.3f}")

    record(0, "init")
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    started = time.perf_counter()

    for epoch in range(1, int(epochs) + 1):
        model.train(); optimizer.zero_grad(set_to_none=True)
        warmup = epoch <= int(warmup_epochs)
        if warmup:
            xi, log_q = model.sample_posterior(int(R_train))
            elbo = model.log_likelihood(xi, force_all_on=True) + model.log_prior(xi) - log_q
        else:
            elbo = model.elbo_draws(int(R_train))["elbo"]
        (-elbo.mean()).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
        optimizer.step()
        if epoch == 1 or epoch % int(record_every) == 0 or epoch == int(warmup_epochs) or epoch == int(epochs):
            record(epoch, "repr" if warmup else "select")

    if DEVICE.type == "cuda": torch.cuda.synchronize()
    train_time = time.perf_counter() - started
    model.eval()
    with torch.no_grad():
        xi_final, _ = model.sample_posterior(int(R_final))
        pred_test = model.decoder(X_test, xi_final)
        fm = metric.function_metrics(signal_test, pred_test)

    result = {"mse": float(fm["mse"]), "r2": float(fm["r2"]), "train_time_sec": float(train_time)}
    feature_pip = unit_pip = None
    if model.decoder.has_feature_gates:
        feature_pip = model.decoder.feature_semantics(xi_final)["active"].float().mean(0).cpu().numpy()
        target = np.asarray(truth["feature_true"], dtype=float).reshape(-1) > 0.5
        selected = feature_pip > float(support_threshold); active, inactive = target, ~target
        ba = np.mean((1.0 - feature_pip[active]) ** 2); b0 = np.mean(feature_pip[inactive] ** 2) if inactive.any() else np.nan
        result.update(tpr=float(selected[active].mean()),
                      auroc=float(roc_auc_score(target.astype(int), feature_pip)) if np.unique(target).size == 2 else np.nan,
                      brier_bal=float(0.5 * (ba + b0)) if inactive.any() else float(ba),
                      expected_support=float(feature_pip.sum()), selected_support=int(selected.sum()),
                      mean_active_pip=float(feature_pip[active].mean()),
                      mean_inactive_pip=float(feature_pip[inactive].mean()) if inactive.any() else np.nan)
    if model.decoder.has_unit_gates:
        unit_pip = model.decoder.unit_semantics(xi_final)["active"].float().mean(0).cpu().numpy()
        result.update(expected_active_units=float(unit_pip.sum()),
                      selected_active_units=int((unit_pip > float(support_threshold)).sum()))
    if selection_mode == "feature_unit_induced_edge":
        result["network_density"] = float(metric.network_density(model.decoder, xi_final))
        result["path_density"] = float(metric.active_path_density(model.decoder, xi_final))

    return {"result": result, "history": pd.DataFrame(history), "model": model, "xi": xi_final.detach(),
            "feature_pip": feature_pip, "unit_pip": unit_pip,
            "init_q0_loc": init_q0_loc, "init_q0_sd": init_q0_sd}


In [4]:
def run_experiment(
    *, n=600, p=100, n_active=10, n_true_units=4, activation="relu", structure="mixed",
    features_per_unit=3, n_interactions=2, n_quadratic=1, extra_scale=0.5,
    target_signal_sd=1.5, sigma2=1.0, x_low=-2.5, x_high=2.5, data_seed=400,
    hidden_dims=(20, 20), selection_mode="feature_unit_induced_edge",
    K_flow=4, flow_hidden_units=128, flow_hidden_layers=2, scale_clip=2.0,
    iaf_ordering_scheme="cyclic3", iaf_shuffle_within_role=True,
    gate_type="normalized_requ", gate_scale=1.0, train_frac=0.8, diagnostic_frac=0.10,
    epochs=1200, warmup_epochs=300, lr=3e-4, R_train=32, R_diag=128, R_final=500,
    record_every=100, init_sd=0.5, init_loc_jitter=0.05, grad_clip=5.0,
    support_threshold=0.5, fit_seed=None, slab_init="auto", slab_sd_ratio=0.1, slab_bias_sd=0.02,
):
    if fit_seed is None: fit_seed = int(data_seed + 100_000 + p + 17 * len(tuple(hidden_dims)))
    X, y, feature_true, signal, info = simfun(
        n=n, p=p, n_active=n_active, n_true_units=n_true_units, activation=activation, structure=structure,
        features_per_unit=features_per_unit, n_interactions=n_interactions, n_quadratic=n_quadratic,
        extra_scale=extra_scale, sigma2=sigma2, target_signal_sd=target_signal_sd,
        x_low=x_low, x_high=x_high, seed=data_seed, device=DEVICE, dtype=DTYPE)

    rng = np.random.default_rng(int(data_seed + 123456)); idx = rng.permutation(int(n))
    n_train = int(round(float(train_frac) * int(n))); train_idx, test_idx = idx[:n_train], idx[n_train:]
    n_diag = max(1, int(round(float(diagnostic_frac) * n_train))); diag_idx = train_idx[:n_diag]
    ti = torch.as_tensor(train_idx, device=DEVICE); di = torch.as_tensor(diag_idx, device=DEVICE); te = torch.as_tensor(test_idx, device=DEVICE)

    print(f"\nExperiment\n  n/p/active : {n}/{p}/{n_active}\n  teacher    : {activation} | {structure} | {n_true_units} units"
          + (f" | {features_per_unit} features/unit" if structure == "mixed" else "")
          + f"\n  extra      : interaction={n_interactions} | quadratic={n_quadratic}"
          + f"\n  fitted BNN : {tuple(hidden_dims)} | {selection_mode}"
          + f"\n  flow       : K={K_flow} | {iaf_ordering_scheme} | slab_init={slab_init}"
          + f"\n  training   : epochs={epochs} | warmup={warmup_epochs} | R={R_train}/{R_diag}/{R_final}"
          + f"\n  seed       : data={data_seed} | fit={fit_seed}")

    out = train_trace(
        X[ti], y[ti], X[di], signal[di], X[te], signal[te], truth=info,
        selection_mode=selection_mode, hidden_dims=tuple(hidden_dims), sigma2=sigma2, init_sd=init_sd,
        K_flow=K_flow, flow_hidden_units=flow_hidden_units, flow_hidden_layers=flow_hidden_layers,
        scale_clip=scale_clip, flow_seed=int(fit_seed + 17), iaf_ordering_scheme=iaf_ordering_scheme,
        iaf_shuffle_within_role=iaf_shuffle_within_role, gate_type=gate_type, gate_scale=gate_scale,
        epochs=epochs, warmup_epochs=warmup_epochs, lr=lr, R_train=R_train, R_diag=R_diag, R_final=R_final,
        record_every=record_every, init_loc_jitter=init_loc_jitter, grad_clip=grad_clip,
        support_threshold=support_threshold, seed=fit_seed, slab_init=slab_init,
        slab_sd_ratio=slab_sd_ratio, slab_bias_sd=slab_bias_sd)
    out["data_info"] = info
    print("\nFinal result")
    display(pd.DataFrame.from_dict(out["result"], orient="index", columns=["value"]).round(6))
    return out

print("simfun, train_trace, run_experiment ready")


simfun, train_trace, run_experiment ready


## One controlled MLP run

In [18]:
exp = run_experiment(
    n=5000, p=500, n_active=10, n_true_units=4,
    activation="relu", structure="mixed", features_per_unit=3,
    n_interactions=2, n_quadratic=1,
    hidden_dims=(20, 20), selection_mode="feature_unit_induced_edge",
    K_flow=6, epochs=1200, warmup_epochs=300, record_every=100,
    data_seed=400, slab_init="auto",
)

display(exp["history"].round(4))


Experiment
  n/p/active : 5000/500/10
  teacher    : relu | mixed | 4 units | 3 features/unit
  extra      : interaction=2 | quadratic=1
  fitted BNN : (20, 20) | feature_unit_induced_edge
  flow       : K=6 | cyclic3 | slab_init=auto
  training   : epochs=1200 | warmup=300 | R=32/128/500
  seed       : data=400 | fit=100934
epoch=0000 init   reprR2=-0.0092 gatedR2=-0.0092 predVar=0.00139 U/V/tauSD=0.00772/0.495/0.473 active=0.487
epoch=0001 repr   reprR2=-0.0010 gatedR2=-0.0010 predVar=0.00131 U/V/tauSD=0.00775/0.498/0.547 active=0.521
epoch=0100 repr   reprR2=+0.6296 gatedR2=+0.6296 predVar=0.109 U/V/tauSD=0.214/0.902/0.982 active=0.514
epoch=0200 repr   reprR2=+0.5466 gatedR2=+0.5466 predVar=0.0722 U/V/tauSD=0.335/0.865/0.955 active=0.517
epoch=0300 repr   reprR2=-4.3316 gatedR2=-4.3316 predVar=1.86 U/V/tauSD=2.68/1.47/1.76 active=0.501
epoch=0400 select reprR2=-865.0767 gatedR2=+0.9385 predVar=3.5e+03 U/V/tauSD=0.804/0.663/0.104 active=0.061
epoch=0500 select reprR2=-476.8272 gate

,value
mse,0.254952
r2,0.887502
train_time_sec,2413.811629
tpr,1.000000
auroc,1.000000
brier_bal,0.000004
expected_support,10.652000
selected_support,10.000000
mean_active_pip,1.000000
mean_inactive_pip,0.001331


,epoch,phase,repr_r2,gated_r2,repr_mse,pred_var,pred_mean_sd,post_U_sd,post_V_sd,post_tau_sd,q0_U_sd,q0_V_sd,q0_tau_sd,margin_mean,margin_sd,active_rate,loglik,logprior,logq,kl_like,feature_pip_mean,active_feature_pip,inactive_feature_pip,unit_pip_mean,expected_units
0,0,init,-0.0092,-0.0092,2.7381,0.0014,0.0306,0.0077,0.4949,0.4731,0.0078,0.5000,0.5000,-0.0176,0.6704,0.4870,-10132.1113,-10192.4824,36391.3555,46583.8359,0.4911,0.4867,0.4912,0.4361,17.4453
1,1,repr,-0.0010,-0.0010,2.7158,0.0013,0.0268,0.0078,0.4976,0.5470,0.0078,0.5002,0.5002,0.0368,0.7145,0.5215,-10098.5293,-10193.2988,36341.8750,46535.1719,0.5269,0.5250,0.5269,0.4537,18.1484
2,100,repr,0.6296,0.6296,1.0049,0.1092,1.1532,0.2139,0.9017,0.9823,0.0078,0.5019,0.5023,0.0391,1.4178,0.5138,-6733.3184,-13099.0947,4831.8979,17930.9922,0.5290,0.5297,0.5290,0.3240,12.9609
3,200,repr,0.5466,0.5466,1.2301,0.0722,0.9395,0.3354,0.8647,0.9547,0.0078,0.5019,0.5022,0.0894,1.4018,0.5170,-7244.1670,-13399.9092,-2023.5575,11376.3516,0.5359,0.5383,0.5358,0.2807,11.2266
4,300,repr,-4.3316,-4.3316,14.4649,1.8571,1.6672,2.6752,1.4720,1.7643,0.0078,0.5019,0.5023,-0.0096,9.2212,0.5010,-42847.4219,-767274.5000,-21501.8867,745772.6250,0.5089,0.2859,0.5135,0.4016,16.0625
5,400,select,-865.0767,0.9385,2349.6975,3498.2122,10.1239,0.8037,0.6628,0.1042,0.0078,0.5020,0.4938,-1.5441,1.0895,0.0606,-5984.4639,-16262.5254,-11617.8145,4644.7109,0.0303,1.0000,0.0105,0.4391,17.5625
6,500,select,-476.8272,0.7555,1296.3624,9258.8555,11.7176,0.6555,0.6865,0.0691,0.0078,0.5019,0.4915,-1.7736,1.0603,0.0475,-6715.7881,-14014.6797,-10326.8115,3687.8682,0.0280,1.0000,0.0082,0.2912,11.6484
7,600,select,-7146.7310,0.9322,19392.0527,60676.2812,25.2193,0.9460,0.7304,0.0567,0.0078,0.5019,0.4910,-2.1226,1.2191,0.0399,-5889.8154,-16767.5605,-14190.0703,2577.4902,0.0239,1.0000,0.0040,0.2393,9.5703
8,700,select,-3066.5886,0.9373,8322.4785,56254.4297,20.9278,0.9176,0.7042,0.0438,0.0078,0.5018,0.4897,-2.3154,1.1529,0.0350,-6018.8193,-16440.2832,-14060.8809,2379.4023,0.0217,1.0000,0.0018,0.2006,8.0234
9,800,select,-2203.6497,0.8365,5981.2944,16923.8379,11.3768,0.7068,0.7194,0.0485,0.0078,0.5017,0.4886,-2.1307,1.1306,0.0371,-6352.0903,-13696.0527,-11449.6396,2246.4131,0.0240,1.0000,0.0041,0.2012,8.0469


In [7]:
# trig + 2 interactions
exp_trig_int = run_experiment(
    n=5000, p=500, n_active=10, n_true_units=4,
    activation="trig", structure="mixed", features_per_unit=3,
    n_interactions=2, n_quadratic=0,
    hidden_dims=(20, 20), selection_mode="feature_unit_induced_edge",
    K_flow=6, epochs=1200, warmup_epochs=300,
    R_train=32, R_diag=128, R_final=500,
    data_seed=400
)


Experiment
  n/p/active : 5000/500/10
  teacher    : trig | mixed | 4 units | 3 features/unit
  extra      : interaction=2 | quadratic=0
  fitted BNN : (20, 20) | feature_unit_induced_edge
  flow       : K=6 | cyclic3 | slab_init=auto
  training   : epochs=1200 | warmup=300 | R=32/128/500
  seed       : data=400 | fit=100934
epoch=0000 init   reprR2=-0.0028 gatedR2=-0.0028 predVar=0.00139 U/V/tauSD=0.00772/0.495/0.473 active=0.487
epoch=0001 repr   reprR2=+0.0027 gatedR2=+0.0027 predVar=0.00127 U/V/tauSD=0.00775/0.498/0.547 active=0.521
epoch=0100 repr   reprR2=+0.0157 gatedR2=+0.0157 predVar=0.423 U/V/tauSD=0.229/0.992/1.16 active=0.518
epoch=0200 repr   reprR2=+0.0950 gatedR2=+0.0950 predVar=0.0895 U/V/tauSD=0.353/0.9/1.12 active=0.496
epoch=0300 repr   reprR2=+0.1008 gatedR2=+0.1008 predVar=0.233 U/V/tauSD=0.555/0.962/0.887 active=0.483
epoch=0400 select reprR2=-130.5868 gatedR2=+0.6195 predVar=7.55e+03 U/V/tauSD=0.808/0.738/0.0705 active=0.039
epoch=0500 select reprR2=-2507.5400 g

,value
mse,0.833669
r2,0.616534
train_time_sec,315.043014
tpr,0.600000
auroc,0.761633
brier_bal,0.200001
expected_support,6.225999
selected_support,6.000000
mean_active_pip,0.600000
mean_inactive_pip,0.000461


In [8]:
exp_trig_int = run_experiment(
    n=5000, p=500, n_active=10, n_true_units=4,
    activation="trig", structure="mixed", features_per_unit=3,
    n_interactions=2, n_quadratic=0,
    hidden_dims=(40, 40), selection_mode="feature_unit_induced_edge",
    K_flow=6, epochs=3000, warmup_epochs=500,
    R_train=32, R_diag=128, R_final=500,
    data_seed=400
)


Experiment
  n/p/active : 5000/500/10
  teacher    : trig | mixed | 4 units | 3 features/unit
  extra      : interaction=2 | quadratic=0
  fitted BNN : (40, 40) | feature_unit_induced_edge
  flow       : K=6 | cyclic3 | slab_init=auto
  training   : epochs=3000 | warmup=500 | R=32/128/500
  seed       : data=400 | fit=100934
epoch=0000 init   reprR2=-0.0028 gatedR2=-0.0028 predVar=0.00219 U/V/tauSD=0.00791/0.494/0.505 active=0.500
epoch=0001 repr   reprR2=+0.0129 gatedR2=+0.0129 predVar=0.00224 U/V/tauSD=0.00794/0.502/0.449 active=0.487
epoch=0100 repr   reprR2=+0.3002 gatedR2=+0.3002 predVar=0.272 U/V/tauSD=0.35/0.929/1.07 active=0.649
epoch=0200 repr   reprR2=+0.3271 gatedR2=+0.3271 predVar=0.152 U/V/tauSD=0.49/0.957/0.757 active=0.664
epoch=0300 repr   reprR2=+0.1985 gatedR2=+0.1985 predVar=0.122 U/V/tauSD=0.548/0.932/0.895 active=0.655
epoch=0400 repr   reprR2=+0.0037 gatedR2=+0.0037 predVar=0.667 U/V/tauSD=1.14/1.17/0.969 active=0.702
epoch=0500 repr   reprR2=+0.1484 gatedR2=+0.1

,value
mse,0.798244
r2,0.632829
train_time_sec,1847.477100
tpr,0.600000
auroc,0.770000
brier_bal,0.201021
expected_support,7.158000
selected_support,7.000000
mean_active_pip,0.600000
mean_inactive_pip,0.002363



## Controlled K-flow comparison

Run this only after the single experiment works. All runs below share the same data, fit seed, slab initialization, and **exact same q0 jitter**. The final check reports the maximum difference between initial `q0.loc` vectors; it should be zero.


In [7]:
depth_runs = {}
for K in (3, 4, 6, 7):
    depth_runs[K] = run_experiment(
        n=600, p=100, n_active=10, n_true_units=4,
        activation="relu", structure="mixed", features_per_unit=3,
        n_interactions=2, n_quadratic=1,
        hidden_dims=(20, 20), selection_mode="feature_unit_induced_edge",
        K_flow=K, epochs=1200, warmup_epochs=300, record_every=100,
        data_seed=400, fit_seed=100534, slab_init="auto",
    )

base = depth_runs[3]["init_q0_loc"]
print("max initial q0.loc difference vs K=3:")
for K, out in depth_runs.items():
    print(f"  K={K}: {(out['init_q0_loc'] - base).abs().max().item():.3e}")

summary = pd.DataFrame({K: out["result"] for K, out in depth_runs.items()}).T
summary.index.name = "K_flow"
display(summary.round(4))

trace = pd.concat([out["history"].assign(K_flow=K) for K, out in depth_runs.items()], ignore_index=True)
display(trace[["K_flow", "epoch", "phase", "repr_r2", "gated_r2", "pred_var", "q0_U_sd", "post_U_sd",
               "post_V_sd", "post_tau_sd", "margin_mean", "active_rate", "feature_pip_mean", "unit_pip_mean"]].round(4))


Experiment
  n/p/active : 600/100/10
  teacher    : relu | mixed | 4 units | 3 features/unit
  extra      : interaction=2 | quadratic=1
  fitted BNN : (20, 20) | feature_unit_induced_edge
  flow       : K=3 | cyclic3 | slab_init=auto
  training   : epochs=1200 | warmup=300 | R=32/128/500
  seed       : data=400 | fit=100534
epoch=0000 init   reprR2=-0.1839 gatedR2=-0.1839 predVar=0.000677 U/V/tauSD=0.017/0.498/0.493 active=0.478
epoch=0001 repr   reprR2=-0.1758 gatedR2=-0.1758 predVar=0.00067 U/V/tauSD=0.0171/0.492/0.491 active=0.501
epoch=0100 repr   reprR2=+0.6117 gatedR2=+0.6117 predVar=0.574 U/V/tauSD=0.257/1.02/1.11 active=0.544
epoch=0200 repr   reprR2=+0.5047 gatedR2=+0.5047 predVar=0.482 U/V/tauSD=0.51/0.963/1.26 active=0.608
epoch=0300 repr   reprR2=+0.4650 gatedR2=+0.4650 predVar=0.622 U/V/tauSD=0.645/0.967/0.92 active=0.582
epoch=0400 select reprR2=-111.3441 gatedR2=-0.1439 predVar=9.55e+03 U/V/tauSD=0.884/0.957/0.601 active=0.422
epoch=0500 select reprR2=-623.0570 gatedR2=

,value
mse,2.078776
r2,-0.000372
train_time_sec,39.451262
tpr,0.200000
auroc,0.262778
brier_bal,0.268876
expected_support,49.675999
selected_support,47.000000
mean_active_pip,0.466400
mean_inactive_pip,0.500133



Experiment
  n/p/active : 600/100/10
  teacher    : relu | mixed | 4 units | 3 features/unit
  extra      : interaction=2 | quadratic=1
  fitted BNN : (20, 20) | feature_unit_induced_edge
  flow       : K=4 | cyclic3 | slab_init=auto
  training   : epochs=1200 | warmup=300 | R=32/128/500
  seed       : data=400 | fit=100534
epoch=0000 init   reprR2=-0.1839 gatedR2=-0.1839 predVar=0.000677 U/V/tauSD=0.017/0.498/0.493 active=0.478
epoch=0001 repr   reprR2=-0.1737 gatedR2=-0.1737 predVar=0.000676 U/V/tauSD=0.0171/0.492/0.491 active=0.500
epoch=0100 repr   reprR2=+0.4823 gatedR2=+0.4823 predVar=0.738 U/V/tauSD=0.282/0.82/0.965 active=0.487
epoch=0200 repr   reprR2=+0.5310 gatedR2=+0.5310 predVar=0.556 U/V/tauSD=0.639/0.978/1.12 active=0.529
epoch=0300 repr   reprR2=+0.5069 gatedR2=+0.5069 predVar=0.832 U/V/tauSD=0.752/0.985/0.968 active=0.514
epoch=0400 select reprR2=-14.3257 gatedR2=+0.5508 predVar=873 U/V/tauSD=0.826/0.715/0.139 active=0.726
epoch=0500 select reprR2=-771.6970 gatedR2=+0

,value
mse,1.515688
r2,0.270603
train_time_sec,47.736421
tpr,1.000000
auroc,0.677778
brier_bal,0.498648
expected_support,99.878006
selected_support,100.000000
mean_active_pip,1.000000
mean_inactive_pip,0.998645



Experiment
  n/p/active : 600/100/10
  teacher    : relu | mixed | 4 units | 3 features/unit
  extra      : interaction=2 | quadratic=1
  fitted BNN : (20, 20) | feature_unit_induced_edge
  flow       : K=6 | cyclic3 | slab_init=auto
  training   : epochs=1200 | warmup=300 | R=32/128/500
  seed       : data=400 | fit=100534
epoch=0000 init   reprR2=-0.1839 gatedR2=-0.1839 predVar=0.000677 U/V/tauSD=0.017/0.498/0.493 active=0.478
epoch=0001 repr   reprR2=-0.1687 gatedR2=-0.1687 predVar=0.000682 U/V/tauSD=0.0171/0.493/0.492 active=0.501
epoch=0100 repr   reprR2=+0.5748 gatedR2=+0.5748 predVar=0.379 U/V/tauSD=0.414/0.864/1.07 active=0.281
epoch=0200 repr   reprR2=+0.5035 gatedR2=+0.5035 predVar=0.465 U/V/tauSD=0.714/0.991/0.978 active=0.377
epoch=0300 repr   reprR2=+0.5740 gatedR2=+0.5740 predVar=0.44 U/V/tauSD=0.791/0.974/0.956 active=0.381
epoch=0400 select reprR2=-70.1348 gatedR2=+0.6464 predVar=663 U/V/tauSD=0.804/0.615/0.0857 active=0.273
epoch=0500 select reprR2=-2410.6858 gatedR2=

...


Experiment
  n/p/active : 600/100/10
  teacher    : relu | mixed | 4 units | 3 features/unit
  extra      : interaction=2 | quadratic=1
  fitted BNN : (20, 20) | feature_unit_induced_edge
  flow       : K=7 | cyclic3 | slab_init=auto
  training   : epochs=1200 | warmup=300 | R=32/128/500
  seed       : data=400 | fit=100534
epoch=0000 init   reprR2=-0.1839 gatedR2=-0.1839 predVar=0.000677 U/V/tauSD=0.017/0.498/0.493 active=0.478
epoch=0001 repr   reprR2=-0.1670 gatedR2=-0.1670 predVar=0.00069 U/V/tauSD=0.0171/0.494/0.492 active=0.501
epoch=0100 repr   reprR2=+0.4662 gatedR2=+0.4662 predVar=0.573 U/V/tauSD=0.452/0.881/0.978 active=0.396
epoch=0200 repr   reprR2=+0.0837 gatedR2=+0.0837 predVar=1.04 U/V/tauSD=0.642/0.999/1.07 active=0.374
epoch=0300 repr   reprR2=+0.5417 gatedR2=+0.5417 predVar=0.507 U/V/tauSD=0.89/1.01/1.02 active=0.524
epoch=0400 select reprR2=-24.4261 gatedR2=+0.6112 predVar=1.06e+03 U/V/tauSD=0.887/0.557/0.115 active=0.364
epoch=0500 select reprR2=-892.7213 gatedR2=+

,value
mse,1.023752
r2,0.507338
train_time_sec,76.447243
tpr,1.000000
auroc,0.955556
brier_bal,0.053289
expected_support,24.715998
selected_support,20.000000
mean_active_pip,1.000000
mean_inactive_pip,0.163511


max initial q0.loc difference vs K=3:
  K=3: 0.000e+00
  K=4: 0.000e+00
  K=6: 0.000e+00
  K=7: 0.000e+00


,mse,r2,train_time_sec,tpr,auroc,brier_bal,expected_support,selected_support,mean_active_pip,mean_inactive_pip,expected_active_units,selected_active_units,network_density,path_density
K_flow,,,,,,,,,,,,,,
3,2.0788,-0.0004,39.4513,0.2,0.2628,0.2689,49.676,47.0,0.4664,0.5001,0.182,0.0,0.0022,0.0000
4,1.5157,0.2706,47.7364,1.0,0.6778,0.4986,99.878,100.0,1.0000,0.9986,4.206,4.0,0.0919,0.0110
6,0.8770,0.5780,70.3398,1.0,0.9778,0.0353,21.034,16.0,1.0000,0.1226,3.510,3.0,0.0223,0.0014
7,1.0238,0.5073,76.4472,1.0,0.9556,0.0533,24.716,20.0,1.0000,0.1635,4.262,4.0,0.0342,0.0022


,K_flow,epoch,phase,repr_r2,gated_r2,pred_var,q0_U_sd,post_U_sd,post_V_sd,post_tau_sd,margin_mean,active_rate,feature_pip_mean,unit_pip_mean
0,3,0,init,-0.1839,-0.1839,0.0007,0.0171,0.0170,0.4976,0.4926,-0.0123,0.4776,0.4870,0.4541
1,3,1,repr,-0.1758,-0.1758,0.0007,0.0172,0.0171,0.4916,0.4907,0.0001,0.5013,0.5143,0.4688
2,3,100,repr,0.6117,0.6117,0.5740,0.0173,0.2569,1.0160,1.1122,0.1552,0.5441,0.6315,0.3258
3,3,200,repr,0.5047,0.5047,0.4815,0.0173,0.5102,0.9631,1.2581,0.3636,0.6081,0.7743,0.1928
4,3,300,repr,0.4650,0.4650,0.6217,0.0173,0.6448,0.9669,0.9202,0.3109,0.5815,0.7542,0.1498
5,3,400,select,-111.3441,-0.1439,9547.5117,0.0174,0.8842,0.9572,0.6010,-0.5220,0.4223,0.5892,0.0049
6,3,500,select,-623.0570,-0.1330,17760.5059,0.0174,0.9284,0.9652,0.6644,-0.6706,0.3888,0.5423,0.0051
7,3,600,select,-35.2612,-0.1383,17870.2129,0.0174,0.9479,0.9760,0.6658,-0.6249,0.4072,0.5683,0.0045
8,3,700,select,-698.6786,-0.1340,16427.3027,0.0174,0.9375,0.9637,0.6456,-0.7082,0.3877,0.5399,0.0070
9,3,800,select,-54.0834,-0.1359,23375.0957,0.0175,0.9721,0.9997,0.6455,-0.6607,0.4179,0.5843,0.0020



# Historical IAF ordering/depth regression test

This section reproduces the original `iaf_ordering_depth_test` setup:

- `n=240`, `p=6`, active features `(0, 3)`
- 2-unit ReLU teacher, fitted shallow network `(5,)`
- 60/20/20 split, data seed 123, fit/flow seed 133
- 2000 epochs, 500 warmup, `R_train=100`, `R_eval=500`, `R_final=5000`
- `scale_clip=1.5`, legacy slab initialization

**Important:** the historical trainer below intentionally uses the old global-RNG jitter and lets evaluation draws consume training RNG. This is only for regression against the old notebook; current experiments above must keep the controlled RNG behavior.


In [8]:
import Python.config as cfg
import Python.utils as ut
import Python.simfun as old_sim

HIST_SEED = 123
X_old, y_old, feature_true_old, unit_true_old, signal_old, info_old = old_sim.simfun_grouped_bnn(
    n=240, p=6, active_features=(0, 3), n_true_units=2, fit_units=5,
    repu_power=None, sigma2=1.0, target_signal_sd=1.5, seed=HIST_SEED, device=DEVICE)

split_cfg = cfg.SplitConfig(train_frac=0.60, val_frac=0.20, test_frac=0.20, seed=HIST_SEED + 1000)
indices = ut.make_split(X_old.shape[0], split_cfg)
splits_old = ut.split_data(X_old, y_old, indices, mode="tensor")
signal_old_split = ut.split_data(X_old, signal_old, indices, mode="tensor")

print("active features:", info_old["active_idx"].tolist())
print("train/val/test:", len(indices["train"]), len(indices["val"]), len(indices["test"]))


active features: [0, 3]
train/val/test: 144 48 48


In [9]:
def train_historical_iaf(K_flow, ordering_scheme):
    # Deliberately mirrors the old training RNG behavior.
    seed = HIST_SEED + 10
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    kwargs = dict(
        X=splits_old["X_train"], y=splits_old["y_train"], input_dim=6, hidden_dims=(5,), out_dim=1,
        selection_mode="feature_group", family=info_old["family"], sigma2=info_old["sigma2"], init_sd=0.5,
        K_flow=int(K_flow), flow_type="iaf", flow_hidden_units=128, flow_hidden_layers=2, scale_clip=1.5,
        flow_seed=seed, iaf_ordering_scheme=ordering_scheme, iaf_shuffle_within_role=True,
        gate_type="normalized_requ", gate_scale=1.0)
    if HAS_SLAB_INIT: kwargs.update(slab_init="legacy")
    model = md.GroupedBNNVI(**kwargs).to(DEVICE)

    # Historical behavior: jitter uses the global RNG after K flow layers have been constructed.
    with torch.no_grad(): model.q0.loc.add_(0.05 * torch.randn_like(model.q0.loc))
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    history = []
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    started = time.perf_counter()

    for epoch in range(1, 2001):
        model.train(); optimizer.zero_grad(set_to_none=True); warmup = epoch <= 500
        if warmup:
            xi, log_q = model.sample_posterior(100)
            elbo = model.log_likelihood(xi, force_all_on=True) + model.log_prior(xi) - log_q
        else:
            elbo = model.elbo_draws(100)["elbo"]
        (-elbo.mean()).backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); optimizer.step()

        if epoch == 1 or epoch % 500 == 0:
            model.eval()
            # Historical behavior: no fork_rng; evaluation advances the training RNG.
            with torch.no_grad():
                xi_eval, _ = model.sample_posterior(500)
                pred = model.decoder(splits_old["X_val"], xi_eval, force_all_on=warmup)
                fm = metric.function_metrics(signal_old_split["y_val"], pred)
            history.append({"epoch": epoch, "phase": "repr" if warmup else "select", "mse": fm["mse"], "r2": fm["r2"]})
            print(f"epoch={epoch:04d} phase={'repr' if warmup else 'select':6s} valMSE={fm['mse']:.5f} valR2={fm['r2']:.4f}")

    if DEVICE.type == "cuda": torch.cuda.synchronize()
    train_time = time.perf_counter() - started
    model.eval()
    with torch.no_grad(): xi_final, _ = model.sample_posterior(5000)
    metrics = metric.evaluate_bnn(model.decoder, xi_final, splits_old["X_test"], signal_old_split["y_test"], info_old)
    result = {"MSE": metrics["mse"], "R2": metrics["r2"], "AUROC": metrics.get("auroc", np.nan),
              "AUPRC": metrics.get("auprc", np.nan), "train_s": train_time}
    orders = model.flow.ordering_summary() if hasattr(model.flow, "ordering_summary") else None
    if orders is not None:
        result["orders"] = " | ".join("<".join(x["role_order"]) for x in orders)
    return {"result": result, "history": pd.DataFrame(history), "model": model, "xi": xi_final}


In [10]:
historical_targets = pd.DataFrame([
    {"configuration": "1L tau-last", "MSE": 0.0564, "R2": 0.9805, "AUROC": 1.0, "AUPRC": 1.0},
    {"configuration": "6L six permutations", "MSE": 0.0553, "R2": 0.9809, "AUROC": 1.0, "AUPRC": 1.0},
    {"configuration": "4L cyclic + tau-last", "MSE": 0.0491, "R2": 0.9830, "AUROC": 1.0, "AUPRC": 1.0},
    {"configuration": "6L two 3-layer cycles", "MSE": 0.0536, "R2": 0.9815, "AUROC": 1.0, "AUPRC": 1.0},
]).set_index("configuration")
print("Historical target from the original notebook")
display(historical_targets)

flow_specs_old = {
    "1L tau-last": (1, "cyclic3"),
    "6L six permutations": (6, "six_permutations"),
    "4L cyclic + tau-last": (4, "cyclic3"),
    "6L two 3-layer cycles": (6, "cyclic3"),
}

historical_runs = {label: train_historical_iaf(K, scheme) for label, (K, scheme) in flow_specs_old.items()}
rerun = pd.DataFrame({label: out["result"] for label, out in historical_runs.items()}).T
rerun.index.name = "configuration"
print("Current code under historical experimental conditions")
display(rerun.round(4))

comparison = historical_targets.join(rerun[["MSE", "R2", "AUROC", "AUPRC"]], lsuffix="_old", rsuffix="_now")
comparison["dR2"] = comparison["R2_now"] - comparison["R2_old"]
comparison["dMSE"] = comparison["MSE_now"] - comparison["MSE_old"]
print("Regression difference")
display(comparison.round(4))

Historical target from the original notebook


,MSE,R2,AUROC,AUPRC
configuration,,,,
1L tau-last,0.0564,0.9805,1.0,1.0
6L six permutations,0.0553,0.9809,1.0,1.0
4L cyclic + tau-last,0.0491,0.9830,1.0,1.0
6L two 3-layer cycles,0.0536,0.9815,1.0,1.0


epoch=0001 phase=repr   valMSE=3.91310 valR2=-1.0134
epoch=0500 phase=repr   valMSE=0.04511 valR2=0.9768
epoch=1000 phase=select valMSE=0.02759 valR2=0.9858
epoch=1500 phase=select valMSE=0.04431 valR2=0.9772
epoch=2000 phase=select valMSE=0.03678 valR2=0.9811
epoch=0001 phase=repr   valMSE=4.56581 valR2=-1.3492
epoch=0500 phase=repr   valMSE=0.04169 valR2=0.9786
epoch=1000 phase=select valMSE=0.04039 valR2=0.9792
epoch=1500 phase=select valMSE=0.03171 valR2=0.9837
epoch=2000 phase=select valMSE=0.03177 valR2=0.9837
epoch=0001 phase=repr   valMSE=4.39348 valR2=-1.2605
epoch=0500 phase=repr   valMSE=0.05089 valR2=0.9738
epoch=1000 phase=select valMSE=0.03088 valR2=0.9841
epoch=1500 phase=select valMSE=0.03150 valR2=0.9838
epoch=2000 phase=select valMSE=0.03029 valR2=0.9844
epoch=0001 phase=repr   valMSE=4.56735 valR2=-1.3500
epoch=0500 phase=repr   valMSE=0.04001 valR2=0.9794
epoch=1000 phase=select valMSE=0.02908 valR2=0.9850
epoch=1500 phase=select valMSE=0.03105 valR2=0.9840
epoch=20

,MSE,R2,AUROC,AUPRC,train_s,orders
configuration,,,,,,
1L tau-last,0.05634,0.980536,1.0,1.0,13.784161,U<V<tau
6L six permutations,0.054554,0.981153,1.0,1.0,36.929147,U<V<tau | V<tau<U | tau<U<V | V<U<tau | U<tau<...
4L cyclic + tau-last,0.046418,0.983964,1.0,1.0,27.396563,U<V<tau | V<tau<U | tau<U<V | U<V<tau
6L two 3-layer cycles,0.052303,0.981931,1.0,1.0,38.464738,U<V<tau | V<tau<U | tau<U<V | U<V<tau | V<tau<...


Regression difference


,MSE_old,R2_old,AUROC_old,AUPRC_old,MSE_now,R2_now,AUROC_now,AUPRC_now,dR2,dMSE
configuration,,,,,,,,,,
1L tau-last,0.0564,0.9805,1.0,1.0,0.05634,0.980536,1.0,1.0,0.000036,-0.00006
6L six permutations,0.0553,0.9809,1.0,1.0,0.054554,0.981153,1.0,1.0,0.000253,-0.000746
4L cyclic + tau-last,0.0491,0.9830,1.0,1.0,0.046418,0.983964,1.0,1.0,0.000964,-0.002682
6L two 3-layer cycles,0.0536,0.9815,1.0,1.0,0.052303,0.981931,1.0,1.0,0.000431,-0.001297



## Optional: recover the original MCMC posterior metrics

The historical target table also contained Active SKL and Zero JS. Those require the original MCMC reference and are much more expensive. Only run the next cell after the VI-only R²/MSE regression above is close to the historical benchmark.


In [ ]:
RUN_MCMC_REFERENCE = False

if RUN_MCMC_REFERENCE:
    import Python.bnn_mcmc as mcmc
    reference_kwargs = dict(
        X=splits_old["X_train"], y=splits_old["y_train"], input_dim=6, hidden_dims=(5,), out_dim=1,
        selection_mode="feature_group", family=info_old["family"], sigma2=info_old["sigma2"],
        init_sd=0.5, K_flow=0, flow_type="meanfield", gate_type="normalized_requ", gate_scale=1.0)
    if HAS_SLAB_INIT: reference_kwargs.update(slab_init="legacy")
    reference_model = md.GroupedBNNVI(**reference_kwargs).to(DEVICE)
    reference = mcmc.run_bnn_mcmc_chains(reference_model, seed=HIST_SEED + 5000, initial_state="prior",
                                          n_chains=8, N=20000, burnin=4000, thin=1, S_max=100, print_every=None)
    reference_xi = torch.as_tensor(reference["xi_draws"], device=DEVICE, dtype=X_old.dtype)
    rows = []
    for label, out in historical_runs.items():
        recovered = metric.evaluate_bnn(out["model"].decoder, out["xi"], splits_old["X_test"], signal_old_split["y_test"], info_old,
                                        reference_decoder=reference_model.decoder, reference_xi=reference_xi)
        rows.append({"configuration": label, "Active SKL": recovered["active_skl"], "Zero JS": recovered["zero_js"]})
    display(pd.DataFrame(rows).set_index("configuration").round(4))